

# Transform Results Data
1. Read bronze `results` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `results` table


- ## Step 1 - Read Bronze Table Data

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.table(bronze_table)

## Step 2 : Keep only columns required for Analytics(Drop URL)

In [0]:
results_selected_df = results_df.drop("url")

## Step 3 &4 - Standardize column name


In [0]:
results_renamed_df = (
    results_selected_df
    .withColumnsRenamed(
        {"constructorId":"constructor_id",
         "driverId":"driver_id",
         "raceName":"race_name",
         "positionText":"finish_position_text",
         "date":"race_date",
         "grid":"grid_position",
         "laps":"completed_laps",
         "number":"car_number",
         "position":"finish_position"
         }
    )
)

In [0]:

# results_renamed_df.count()

## Step 5 - Filter out rows where primary key is NULL

season, round, custructor_id or driver_id

In [0]:
results_valid_df = (
    results_renamed_df
    .filter(
        F.col("season").isNotNull() |
        F.col("round").isNotNull() |
        F.col("constructor_id").isNotNull()|
        F.col("driver_id").isNotNull()

    )
)


In [0]:
# results_valid_df.count()

## Step 6 - Remove Duplicates

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season","round","constructor_id","driver_id"])

In [0]:
# results_distinct_df.count()

## Step 7 - Transform required column values to titlecase

In [0]:
results_final_df = (
    results_distinct_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
# display(results_final_df)

## Step 8 - Write data to silver table

In [0]:
(
    results_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
# display(spark.table(silver_table))